> ⚠️ **AIHub 는 해외 IP 에서의 다운로드를 막습니다. Colab 서버는 해외라 이 노트북의 다운로드 단계는 실패합니다.**
> 국내 PC(또는 국내 서버)에서 받아 `scripts/merge_parts.py` 와 `scripts/prepare_aihub71363.py` 로 변환한 뒤,
> 결과를 Drive 에 올려 `notebooks/colab_train.ipynb` 로 학습하십시오. 아래 병합·확인 셀은 국내 리눅스 환경에서 그대로 쓸 수 있습니다.

# AIHub 국립공원 변화탐지(71363) → Colab 에서 바로 받기

전체 98GB 를 받지 않습니다. 주제3(0.55~0.7m 위성)에 맞는 것은 **Skyset(0.5m 위성)** 뿐입니다.
Drone(0.1m)은 해상도가 너무 높고, Sentinel2(10m)·Landsat(30m)는 너무 낮아 쓰지 않습니다.

| 단계 | 파일 | 크기 | key |
|---|---|---|---|
| **1. 형식 확인** | 01.메타데이터_02.Skyset | 10 MB | 533615 |
| | VL_01.LABEL_02.Skyset (검증 라벨) | 16 MB | 491178 |
| | VL_02.JSON_02.Skyset (검증 JSON) | 110 MB | 491182 |
| **2. 작게 시작** | VS_02.Skyset (검증 원천) | 5.5 GB | 491174 |
| **3. 본격 학습** | TL_01.LABEL_02 / TL_02.JSON_02 (학습 라벨) | 94 MB / 545 MB | 491166, 491170 |
| | TS_02.Skyset (학습 원천) | 42.9 GB | 491162 |
| (참고) | 02.SHP_02.Skyset | 726 MB | 533619 |

**1단계 결과(라벨 형식)를 먼저 확인한 뒤** 2·3단계로 갑니다. 한 시점 토지피복 라벨인지, 전/후 변화 라벨인지에 따라 변환 방식이 달라집니다.

준비물: AIHub 에서 데이터셋 **다운로드 승인** + 마이페이지의 **API 키**.
Colab 디스크는 보통 여유가 80GB 안팎이라 TS_02(43GB)는 받고 → 풀고 → 256 패치로 변환 → 원본 삭제 순서로 처리해야 합니다.
변환된 패치만 Google Drive 에 저장하면 세션이 끊겨도 다시 받을 필요가 없습니다.

In [ ]:
# [1] Drive 연결 + 작업 폴더 (변환 결과만 Drive 에 저장, 원본은 Colab 로컬 디스크)
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/aihub'          # 원본 (빠른 로컬 디스크, 세션 끝나면 사라짐)
OUT = '/content/drive/MyDrive/knps/data'   # 변환된 학습 패치 (보존)
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT, exist_ok=True)
%cd {WORK}
!df -h /content | tail -1

In [ ]:
# [2] AIHub 다운로드 도구 설치
!curl -s -o aihubshell https://api.aihub.or.kr/api/aihubshell.do && chmod +x aihubshell
!./aihubshell -help | head -40      # 옵션 이름은 도구 버전에 따라 다를 수 있으니 여기서 확인

In [ ]:
# [3] API 키 입력 (노트북에 적지 않음)
from getpass import getpass
AIHUB_KEY = getpass('AIHub API 키: ').strip().strip("'\"")   # 복사할 때 딸려온 공백·따옴표 제거
print('키 길이', len(AIHUB_KEY), '| 공백 포함' if any(ch.isspace() for ch in AIHUB_KEY) else '| 공백 없음')
DATASET = '71363'
# 파일 목록 확인
!./aihubshell -mode l -datasetkey {DATASET} | head -80

In [ ]:
# [4] 다운로드 - 단계별로 FILEKEYS 를 바꿔 실행
FILEKEYS = '533615,491178,491182'      # 1단계: 메타데이터 + 검증 라벨/JSON (약 140MB)
# FILEKEYS = '491174'                  # 2단계: 검증 원천 5.5GB
# FILEKEYS = '491166,491170'           # 3단계: 학습 라벨/JSON
# FILEKEYS = '491162'                  # 3단계: 학습 원천 42.9GB (디스크 여유 확인 후)
!./aihubshell -mode d -datasetkey {DATASET} -filekey {FILEKEYS} -aihubapikey '{AIHUB_KEY}'
!df -h /content | tail -1

**`인증실패, 권한이 거부되었습니다` 가 나오면**
1. 71363 데이터셋 페이지에서 **다운로드 신청 → 마이페이지에서 승인 완료** 상태인지 확인 (승인 전에는 API 다운로드 불가)
2. 비밀번호가 아니라 **마이페이지에서 발급한 API 키**인지, 승인받은 **같은 계정**의 **가장 최근** 키인지 확인
3. [3] 셀을 다시 실행해 키를 새로 입력 (공백은 자동 제거)
4. 아래 셀로 작은 파일 하나(메타데이터 10MB)만 시험

In [ ]:
# 진단: 작은 파일 하나로 인증만 시험
!./aihubshell -help 2>&1 | grep -i -E "key|mode|file" | head -20
!./aihubshell -mode d -datasetkey {DATASET} -filekey 533615 -aihubapikey '{AIHUB_KEY}' 2>&1 | tail -5

In [ ]:
# [5] 분할 파일(.zip.partN) 병합 -> 압축 해제 -> 조각 삭제 (AIHub 안내의 find|sort|cat 과 같은 동작)
import re, zipfile, glob
from pathlib import Path

def merge_and_extract(root=WORK, extract=True):
    groups = {}
    for p in Path(root).rglob('*.part*'):
        m = re.match(r'(.*)\.part(\d+)$', str(p))
        if m:
            groups.setdefault(m.group(1), []).append((int(m.group(2)), p))
    for target, parts in groups.items():
        parts.sort()
        with open(target, 'wb') as out:
            for _, p in parts:
                with open(p, 'rb') as f:
                    while chunk := f.read(1 << 24):
                        out.write(chunk)
        size = os.path.getsize(target)
        print(f'병합 {Path(target).name}: 조각 {len(parts)}개 -> {size / 2**20:.1f} MB')
        if size == 0:
            raise RuntimeError('병합 결과가 0바이트입니다. 경로를 확인하세요')
        for _, p in parts:
            p.unlink()                                  # 디스크 절약
    if extract:
        for z in Path(root).rglob('*.zip'):
            dst = z.with_suffix('')
            if not dst.exists():
                print('압축 해제', z.name)
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(dst)
                z.unlink()                              # 원본 zip 삭제로 디스크 절약

merge_and_extract()
!df -h /content | tail -1

In [ ]:
# [6] 형식 확인 - 폴더 구조, 파일 종류, JSON 샘플
from collections import Counter
import json
files = [p for p in Path(WORK).rglob('*') if p.is_file()]
print('파일 수', len(files)); print(Counter(p.suffix.lower() for p in files).most_common(10))
for d in sorted({p.parent for p in files})[:15]:
    print(d.relative_to(WORK), len(list(d.iterdir())))
js = [p for p in files if p.suffix.lower() == '.json']
if js:
    print('\n=== JSON 샘플:', js[0].relative_to(WORK))
    print(json.dumps(json.load(open(js[0], encoding='utf-8')), ensure_ascii=False, indent=1)[:3000])

In [ ]:
# [7] 샘플을 Drive 로 복사 -> Claude 에게 올려서 변환 스크립트를 맞춤
#     (JSON 3개 + 라벨 이미지 2개 + 메타데이터 일부, 30MB 이하)
import shutil
SAMPLE = Path(OUT).parent / 'aihub_sample'; SAMPLE.mkdir(parents=True, exist_ok=True)
picked = js[:3] + [p for p in files if p.suffix.lower() in ('.png', '.tif', '.tiff')][:2] \
         + [p for p in files if 'meta' in str(p).lower() or '메타' in str(p)][:2]
for p in picked:
    shutil.copy(p, SAMPLE / p.name)
    print(p.relative_to(WORK), f'{p.stat().st_size / 2**20:.2f} MB')
print('Drive 경로:', SAMPLE)